In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q sinling
import os
import json
import random
import numpy as np
from pathlib import Path
from sinling import SinhalaTokenizer, POSTagger
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
random.seed(42)
np.random.seed(42)

print(" Libraries imported successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.2 MB/s eta 0:00:00
 Libraries imported successfully


In [ ]:
# Define paths
BASE_PATH = "YOUR_PATH"
TRAIN_PATH = f"{BASE_PATH}/YOUR_PATH"
OUTPUT_PATH = f"{BASE_PATH}/YOUR_PATH"
MODEL_PATH = f"{BASE_PATH}/YOUR_PATH"

# Verify paths exist
assert os.path.exists(TRAIN_PATH), f"Training path not found: {TRAIN_PATH}"
assert os.path.exists(OUTPUT_PATH), f"Output path not found: {OUTPUT_PATH}"
assert os.path.exists(MODEL_PATH), f"Model path not found: {MODEL_PATH}"

print("  Paths verified:")
print(f"   Train: {TRAIN_PATH}")
print(f"   Output: {OUTPUT_PATH}")
print(f"   Model: {MODEL_PATH}")

In [ ]:
print("Initializing Sinhala NLP tools...")
tokenizer_sin = SinhalaTokenizer()
pos_tagger = POSTagger()

print(" Sinhala tools initialized")

Initializing Sinhala NLP tools...
 Sinhala tools initialized


In [ ]:
# Common Sinhala legal keywords
LEGAL_KEYWORDS = [
    'අධිකරණය',
    'නීතිය',
    'විනිශ්චය',
    'දඬුවම',
    'චෝදනා',
    'සාක්ෂි',
    'තීන්දුව',
    'අභියාචනය',
    'නඩුව',
    'වරද',
    'දෝෂ',
    'බලය',
    'කොන්දේසි',
    'ප්‍රතිපාදන',
    'ව්‍යවස්ථා',
    'පනත',
    'පැමිණිලිකරු',
    'කොන්ත්‍රාත්තුව',
    'පැමිණිල්ල'
]

print(f"  Loaded {len(LEGAL_KEYWORDS)} legal keywords")
print(f"   Examples: {LEGAL_KEYWORDS[:5]}")

  Loaded 19 legal keywords
   Examples: ['අධිකරණය', 'නීතිය', 'විනිශ්චය', 'දඬුවම', 'චෝදනා']


In [ ]:
print("Loading training documents...")

train_files = [f for f in os.listdir(TRAIN_PATH) if f.endswith('.txt')]
print(f" Found {len(train_files)} training documents")

# Load all documents
all_documents = []
failed_loads = 0

for filename in tqdm(train_files, desc="Loading documents"):
    filepath = os.path.join(TRAIN_PATH, filename)
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read().strip()
            if len(text) > 100:  # Only keep documents with meaningful content
                all_documents.append({
                    'filename': filename,
                    'text': text
                })
    except Exception as e:
        failed_loads += 1

print(f" Loaded {len(all_documents)} documents successfully")
if failed_loads > 0:
    print(f"  Failed to load {failed_loads} documents")

Loading training documents...
 Found 964 training documents


Loading documents: 100%|██████████| 964/964 [13:34<00:00,  1.18it/s]

 Loaded 964 documents successfully


In [ ]:
import re

print("Extracting sentences from documents...")

all_sentences = []

# regex: split on . ! ? ;
split_pattern = re.compile(r'[.!?;]+')

for doc in tqdm(all_documents, desc="Extracting sentences"):
    sentences = [
        s.strip()
        for s in split_pattern.split(doc['text'])
        if len(s.strip()) > 20
    ]

    for sent in sentences:
        all_sentences.append({
            'text': sent,
            'source_doc': doc['filename']
        })

print(f" Extracted {len(all_sentences)} sentences total")
print(f"   Average: {len(all_sentences) // len(all_documents)} sentences per document")


Extracting sentences from documents...


Extracting sentences: 100%|██████████| 964/964 [00:00<00:00, 1597.66it/s]

 Extracted 50965 sentences total
   Average: 52 sentences per document


In [ ]:
import re

print("\n" + "="*60)
print("GENERATING POSITIVE SAMPLES")
print("="*60)

# Define sentence splitting pattern (split on . ! ? ;)
split_pattern = re.compile(r'[.!?;]+')

positive_samples = []

# Strategy 1: First sentence of each document (often contains main topic)
print("\n Extracting first sentences...")
for doc in all_documents:
    sentences = [s.strip() for s in split_pattern.split(doc['text']) if len(s.strip()) > 20]
    if len(sentences) > 0:
        positive_samples.append({
            'text': sentences[0],
            'source': 'first_sentence',
            'label': 1
        })

print(f"   ✓ Added {len([s for s in positive_samples if s['source'] == 'first_sentence'])} first sentences")

# Strategy 2: Last sentence of each document (often contains conclusion)
print("\n Extracting last sentences...")
initial_count = len(positive_samples)
for doc in all_documents:
    sentences = [s.strip() for s in split_pattern.split(doc['text']) if len(s.strip()) > 20]
    if len(sentences) > 1:  # Only if doc has multiple sentences
        positive_samples.append({
            'text': sentences[-1],
            'source': 'last_sentence',
            'label': 1
        })

print(f"   ✓ Added {len(positive_samples) - initial_count} last sentences")

# Strategy 3: Sentences containing legal keywords
print("\n Extracting sentences with legal keywords...")
initial_count = len(positive_samples)

for sent_data in all_sentences:
    sent_text = sent_data['text']
    # Check if sentence contains any legal keyword
    if any(keyword in sent_text for keyword in LEGAL_KEYWORDS):
        # Avoid duplicates
        if sent_text not in [s['text'] for s in positive_samples]:
            positive_samples.append({
                'text': sent_text,
                'source': 'legal_keyword',
                'label': 1
            })

print(f"   ✓ Added {len(positive_samples) - initial_count} keyword-rich sentences")

# Strategy 4: Longer sentences (often more informative)
print("\n Extracting longer sentences...")
initial_count = len(positive_samples)

for sent_data in all_sentences:
    sent_text = sent_data['text']
    # If sentence is longer than 50 characters
    if len(sent_text) > 50 and sent_text not in [s['text'] for s in positive_samples]:
        positive_samples.append({
            'text': sent_text,
            'source': 'long_sentence',
            'label': 1
        })
        # Limit to avoid too many samples
        if len(positive_samples) >= 5000:
            break

print(f"    Added {len(positive_samples) - initial_count} longer sentences")

print(f"\n Total positive samples: {len(positive_samples)}")


GENERATING POSITIVE SAMPLES

 Extracting first sentences...
   ✓ Added 964 first sentences

 Extracting last sentences...
   ✓ Added 964 last sentences

 Extracting sentences with legal keywords...
   ✓ Added 17531 keyword-rich sentences

 Extracting longer sentences...
    Added 1 longer sentences

 Total positive samples: 19460


In [ ]:
print("\nDefining negative sample generation functions...")

def shuffle_words(sentence):
    """Randomly shuffle words in a sentence"""
    tokens = tokenizer_sin.tokenize(sentence)
    if len(tokens) > 2:
        random.shuffle(tokens)
        return ' '.join(tokens)
    return sentence

def replace_random_words(sentence, replace_prob=0.3):
    """Replace random words with other words from vocabulary"""
    tokens = tokenizer_sin.tokenize(sentence)
    if len(tokens) < 3:
        return sentence

    # Build vocabulary from all sentences (sample for speed)
    vocab = set()
    for sent in random.sample(all_sentences, min(1000, len(all_sentences))):
        vocab.update(tokenizer_sin.tokenize(sent['text']))
    vocab = list(vocab)

    # Replace words
    new_tokens = []
    for token in tokens:
        if random.random() < replace_prob and len(vocab) > 0:
            new_tokens.append(random.choice(vocab))
        else:
            new_tokens.append(token)

    return ' '.join(new_tokens)

def delete_random_words(sentence, delete_prob=0.3):
    """Delete random words from sentence"""
    tokens = tokenizer_sin.tokenize(sentence)
    if len(tokens) < 4:
        return sentence

    new_tokens = [token for token in tokens if random.random() > delete_prob]

    # Ensure at least 2 words remain
    if len(new_tokens) < 2:
        new_tokens = tokens[:2]

    return ' '.join(new_tokens)

def truncate_sentence(sentence):
    """Truncate sentence to first half"""
    tokens = tokenizer_sin.tokenize(sentence)
    if len(tokens) > 4:
        half = len(tokens) // 2
        return ' '.join(tokens[:half])
    return sentence

print(" Negative sample functions defined")


Defining negative sample generation functions...
 Negative sample functions defined


In [ ]:
print("\n" + "="*60)
print("GENERATING NEGATIVE SAMPLES")
print("="*60)

negative_samples = []

# We want roughly equal number of positive and negative samples
target_negative_count = len(positive_samples)

# Sample sentences to corrupt
sentences_to_corrupt = random.sample(
    [s['text'] for s in positive_samples],
    min(target_negative_count, len(positive_samples))
)

print(f"\nGenerating {target_negative_count} negative samples...")

corruption_methods = [
    ('word_shuffle', shuffle_words),
    ('word_replace', replace_random_words),
    ('word_delete', delete_random_words),
    ('truncate', truncate_sentence)
]

for i, sent in enumerate(tqdm(sentences_to_corrupt, desc="Corrupting sentences")):
    # Choose random corruption method
    method_name, method_func = random.choice(corruption_methods)

    try:
        corrupted = method_func(sent)

        # Make sure corrupted is different from original
        if corrupted != sent and len(corrupted) > 10:
            negative_samples.append({
                'text': corrupted,
                'source': method_name,
                'label': 0
            })
    except Exception as e:
        # If corruption fails, try simple shuffle
        try:
            corrupted = shuffle_words(sent)
            if corrupted != sent:
                negative_samples.append({
                    'text': corrupted,
                    'source': 'fallback_shuffle',
                    'label': 0
                })
        except:
            continue

print(f"\n Generated {len(negative_samples)} negative samples")


GENERATING NEGATIVE SAMPLES

Generating 19460 negative samples...


Corrupting sentences: 100%|██████████| 19460/19460 [04:55<00:00, 65.91it/s]


 Generated 19424 negative samples


In [ ]:
print("\n" + "="*60)
print("CREATING FINAL TRAINING DATASET")
print("="*60)

# Combine positive and negative samples
training_data = positive_samples + negative_samples

# Shuffle
random.shuffle(training_data)

print(f"\n Dataset composition:")
print(f"   Positive samples: {len(positive_samples)} (label=1)")
print(f"   Negative samples: {len(negative_samples)} (label=0)")
print(f"   Total samples: {len(training_data)}")

# Calculate class balance
pos_ratio = len(positive_samples) / len(training_data) * 100
print(f"   Class balance: {pos_ratio:.1f}% positive, {100-pos_ratio:.1f}% negative")


CREATING FINAL TRAINING DATASET

 Dataset composition:
   Positive samples: 19460 (label=1)
   Negative samples: 19424 (label=0)
   Total samples: 38884
   Class balance: 50.0% positive, 50.0% negative


In [ ]:
# Save training data
output_file = f"{OUTPUT_PATH}/Data/training_data.json"

print(f"\nSaving training data to {output_file}...")

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(training_data, f, ensure_ascii=False, indent=2)

print(f" Training data saved successfully")
print(f"   File size: {os.path.getsize(output_file) / 1024 / 1024:.2f} MB")


Saving training data to /content/drive/MyDrive/SinBrief/Outputs/Data/training_data.json...
 Training data saved successfully
   File size: 25.17 MB


In [ ]:
# Save some examples for inspection
print("\nSaving sample examples...")

examples = {
    'positive_examples': positive_samples[:10],
    'negative_examples': negative_samples[:10]
}

examples_file = f"{OUTPUT_PATH}/Data/sample_examples.json"

with open(examples_file, 'w', encoding='utf-8') as f:
    json.dump(examples, f, ensure_ascii=False, indent=2)

print(f" Sample examples saved to {examples_file}")

# Print some examples
print("\n" + "="*60)
print("SAMPLE POSITIVE EXAMPLES:")
print("="*60)
for i, ex in enumerate(examples['positive_examples'][:3], 1):
    print(f"\n{i}. [{ex['source']}]")
    print(f"   {ex['text'][:150]}...")

print("\n" + "="*60)
print("SAMPLE NEGATIVE EXAMPLES:")
print("="*60)
for i, ex in enumerate(examples['negative_examples'][:3], 1):
    print(f"\n{i}. [{ex['source']}]")
    print(f"   {ex['text'][:150]}...")


Saving sample examples...
 Sample examples saved to /content/drive/MyDrive/SinBrief/Outputs/Data/sample_examples.json

SAMPLE POSITIVE EXAMPLES:

1. [first_sentence]
   ශ්‍රී ලංකා ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
2006 අංක 49 දරන එකතු කළ අගය මත බදු
(සංශෝධන) පනත

[සහතිකය සටහත් කළේ 2006 දෙසැම්බර් මස 15 වන ද...

2. [first_sentence]
   ශ්‍රී ලංකා ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
2007 අංක 30 දරන පළාත් පාලන ආයතන
(විශේෂ විධිවිධාන) පනත

[සහතිකය සටහන් කළේ 2007 අගෝස්තු මස 08 ...

3. [first_sentence]
   ශ්‍රී ලංකා
ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
1987 අංක 20 දරන
පළාත් පාලන ආයතන ඡන්ද විමසීම්
(සංශෝධන) පනත

[සහතිකය සටහන් කළේ 1987 අප්‍රේල් ම...

SAMPLE NEGATIVE EXAMPLES:

1. [word_delete]
   පුනරුත්ථාපන සංස්ථාගත කිරීමේ සංශෝධනය කිරීම සඳහා පනතකි...

2. [word_shuffle]
   පත්‍රය සමාජවාදී 01 ශ්‍රී ප්‍රජාතාන්ත්‍රික ගැසට් 2012 ජනරජයේ කොටස වන ( වන අතිරේකය දින II මස ජුනි ලංකා 2012...

3. [word_delete]
   ප්‍රධාන ප්‍රඥප්තියේ 158 වරදකරු කරනු විට රුපියල් පනහකට නොවැඩි ඔහු

In [ ]:
print("\n" + "="*60)
print(" TRAINING DATA GENERATION COMPLETE")
print("="*60)

print("\n Generated:")
print(f"   ✓ {len(positive_samples)} positive samples")
print(f"   ✓ {len(negative_samples)} negative samples")
print(f"   ✓ {len(training_data)} total training samples")

print("\n Files created:")
print(f"   ✓ {OUTPUT_PATH}/Data/training_data.json")
print(f"   ✓ {OUTPUT_PATH}/sample_examples.json")

print("="*60)


 TRAINING DATA GENERATION COMPLETE

 Generated:
   ✓ 19460 positive samples
   ✓ 19424 negative samples
   ✓ 38884 total training samples

 Files created:
   ✓ /content/drive/MyDrive/SinBrief/Outputs/Data/training_data.json
   ✓ /content/drive/MyDrive/SinBrief/Outputs/sample_examples.json
